# VibeVoice ASR and speaker turns

Run top to bottom in a **fresh local Jupyter kernel** with the matching route environment from README. Put one extracted audio/video file path in ignored `local_config.json` or set `TRAUMA_INPUT_FILE`. `DENOISE=True` applies FFmpeg `afftdn`; `False` copies the normalized WAV unchanged. Both versions and the input hash are saved under `runs/`. Section 2 provides short inline audio players and a full-recording HTML comparison.

Outputs are draft anonymous speaker labels, not verified identities or clinical roles. Check the audio before interpreting them. No input recordings or outputs belong in Git.


## 1. Settings and GPU check

In [ ]:
ROUTE = "vibevoice"
from pathlib import Path
import sys
import json
import os
from pipeline_common import (prepare_audio, gpu_preflight, save_turns, save_summary,
                             write_audio_preview, make_inline_audio_clips)

# Prefer an environment variable or ignored local_config.json to keep private paths out of Git.
INPUT_FILE = os.environ.get("TRAUMA_INPUT_FILE", "")
if not INPUT_FILE and Path("local_config.json").is_file():
    INPUT_FILE = json.loads(Path("local_config.json").read_text(encoding="utf-8")).get("input_file", "")
OUTPUT_ROOT = "runs"
AUDIO_STREAM = 0
DENOISE = False  # True / False
NOISE_REDUCTION_DB = 12  # FFmpeg afftdn nr; 1..30; only used if DENOISE=True
PREVIEW_START_SECONDS = 0  # choose where to listen in the recording
PREVIEW_DURATION_SECONDS = 20  # 1..60; bounded notebook audio output

if not INPUT_FILE:
    raise ValueError("Set TRAUMA_INPUT_FILE or create ignored local_config.json with input_file")
print("Notebook Python:", sys.executable)
print("Route:", ROUTE, "| denoise:", DENOISE)


## 2. Convert video/audio to WAV; optionally denoise

In [ ]:
run_dir, model_wav, manifest = prepare_audio(
    INPUT_FILE, OUTPUT_ROOT, ROUTE, denoise=DENOISE,
    audio_stream=AUDIO_STREAM, noise_reduction_db=NOISE_REDUCTION_DB,
)
duration_seconds = manifest["duration_seconds"]
print(f"Model input: {model_wav} ({duration_seconds:.2f}s)")
print(f"Denoising: {manifest['denoise_filter'] or 'off'}")
print(f"Output: {run_dir}")
preview_path = write_audio_preview(run_dir, denoise=DENOISE)
print(f"Open this file in a browser to compare both audio players: {preview_path}")
preview_original, preview_model = make_inline_audio_clips(
    run_dir, start_seconds=PREVIEW_START_SECONDS,
    duration_seconds=PREVIEW_DURATION_SECONDS,
)
from IPython.display import Audio, Markdown, display
display(Markdown(f"**Original audio** — {PREVIEW_START_SECONDS}s to "
                 f"{PREVIEW_START_SECONDS + PREVIEW_DURATION_SECONDS}s"))
display(Audio(filename=str(preview_original)))
display(Markdown(f"**Model input** — denoise {'on' if DENOISE else 'off'}"))
display(Audio(filename=str(preview_model)))


## 3. Infer and save results

In [ ]:
import json
import time
device, gpu_info = gpu_preflight()
print(gpu_info)
import torch
from vibevoice.modular.modeling_vibevoice_asr import VibeVoiceASRForConditionalGeneration
from vibevoice.processor.vibevoice_asr_processor import VibeVoiceASRProcessor
from pipeline_common import parse_time

MODEL_ID = "microsoft/VibeVoice-ASR"
MAX_NEW_TOKENS = 8192
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
processor = VibeVoiceASRProcessor.from_pretrained(MODEL_ID)
model = VibeVoiceASRForConditionalGeneration.from_pretrained(
    MODEL_ID, dtype=dtype, attn_implementation="sdpa", trust_remote_code=True,
).to(device).eval()
inputs = processor(audio=[str(model_wav)], sampling_rate=None,
                   return_tensors="pt", padding=True, add_generation_prompt=True)
inputs = {key: value.to(device) if isinstance(value, torch.Tensor) else value
          for key, value in inputs.items()}
prompt_length = inputs["input_ids"].shape[1]
started = time.perf_counter()
with torch.inference_mode():
    output_ids = model.generate(
        **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
        pad_token_id=processor.pad_id,
        eos_token_id=processor.tokenizer.eos_token_id,
    )
elapsed = time.perf_counter() - started
generated_ids = output_ids[0, prompt_length:]
raw_text = processor.decode(generated_ids, skip_special_tokens=True)
(run_dir / "vibevoice_raw.txt").write_text(raw_text, encoding="utf-8")
parsed = processor.post_process_transcription(raw_text)
(run_dir / "vibevoice_raw.json").write_text(json.dumps(parsed, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
segments = []
parse_failures = []
for index, segment in enumerate(parsed):
    try:
        segments.append(dict(start_seconds=parse_time(segment["start_time"]),
                             end_seconds=parse_time(segment["end_time"]),
                             model_speaker=str(segment.get("speaker_id") or "UNKNOWN"),
                             text=str(segment.get("text") or "")))
    except (KeyError, ValueError, TypeError) as exc:
        parse_failures.append({"index": index, "error": str(exc), "raw": segment})
(run_dir / "parse_failures.json").write_text(json.dumps(parse_failures, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
rows = save_turns(run_dir, segments, duration_seconds)
save_summary(run_dir, model=MODEL_ID, **gpu_info, denoise=DENOISE,
             generated_tokens=len(generated_ids), max_new_tokens=MAX_NEW_TOKENS,
             inference_seconds=round(elapsed, 2), parsed_turns=len(rows),
             parse_failures=len(parse_failures),
             needs_review=not rows or bool(parse_failures) or len(generated_ids) >= MAX_NEW_TOKENS)
print(f"{len(rows)} turns saved in {run_dir}; parse failures: {len(parse_failures)}")
rows[:10]


## 4. Three-panel waveform, speaker activity, and transcript timeline

In [ ]:
from pipeline_common import render_speaker_timeline, show_figure
fig = render_speaker_timeline(
    run_dir, rows, route="vibevoice", source_name=Path(INPUT_FILE).name,
)
show_figure(fig)
